In [ ]:
# Available backbones for water-stress lavender classification:
#   efficientnet_b0   – compact EfficientNet; strong baseline on small plant datasets
#   mobilenet_v3_large – lightweight with squeeze-and-excitation blocks; fast, agriculture-friendly
#   resnet50          – classic workhorse; dominant in plant-disease literature (PlantVillage etc.)
#   densenet121       – dense feature reuse; captures color/texture cues critical for stress detection
#   convnext_tiny     – modern SOTA CNN (2022); excels at fine-grained biological visual tasks

# Set BACKBONES_TO_RUN to a subset (or all) to compare.  To run a single backbone
# just leave a one-element list, e.g.  BACKBONES_TO_RUN = ["efficientnet_b0"]
BACKBONES_TO_RUN = [
    "efficientnet_b0",
    "mobilenet_v3_large",
    "resnet50",
    "densenet121",
    "convnext_tiny",
]


In [13]:
# Imports, reproducibility, and dataset statistics

import os
import random
from pathlib import Path
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, Subset
    from torchvision import datasets, transforms, models
except ImportError as exc:
    raise ImportError(
        "PyTorch and torchvision are required. Install them in your notebook environment before running this notebook."
    ) from exc

import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, f1_score

SEED = 42
DATA_DIR = "lavender_dataset_splitted_stress"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def compute_dataset_mean_std(data_dir, img_size=224, batch_size=8):
    dataset = datasets.ImageFolder(
        root=data_dir,
        transform=transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ]),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    mean = torch.zeros(3)
    std = torch.zeros(3)
    total = 0

    for images, _ in loader:
        b = images.size(0)
        images = images.view(b, images.size(1), -1)
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        total += b

    mean /= total
    std /= total
    return mean, std, dataset.classes, len(dataset)

if not os.path.isdir(DATA_DIR):
    raise SystemExit(f"Dataset directory not found: {DATA_DIR}")

mean, std, class_names, dataset_size = compute_dataset_mean_std(DATA_DIR, img_size=224)
print(f"Dataset size: {dataset_size}")
print(f"Classes: {class_names}")
print(f"Dataset mean: {mean}")
print(f"Dataset std: {std}")

# ImageNet statistics are used for pretrained transfer learning transforms.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
print(f"ImageNet normalization: mean={IMAGENET_MEAN}, std={IMAGENET_STD}")


Dataset size: 118
Classes: ['not_stressed', 'stressed']
Dataset mean: tensor([0.2229, 0.2028, 0.1973])
Dataset std: tensor([0.1884, 0.1824, 0.1612])
ImageNet normalization: mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)


In [ ]:
# Transfer-learning model, freezing policy, loaders, and train/eval utilities

if "BACKBONES_TO_RUN" not in globals():
    BACKBONES_TO_RUN = ["efficientnet_b0"]


# ── backbone helpers ────────────────────────────────────────────────────────

def _get_head_module(model, backbone_name: str):
    """Return the classification head module for the given backbone."""
    if backbone_name in ("efficientnet_b0", "mobilenet_v3_large", "densenet121", "convnext_tiny"):
        return model.classifier
    if backbone_name == "resnet50":
        return model.fc
    raise ValueError(f"Unknown backbone: {backbone_name!r}")


def _get_last_feature_blocks(model, backbone_name: str):
    """Return a list of the last feature-extraction block(s) to unfreeze in Stage 2."""
    if backbone_name in ("efficientnet_b0", "mobilenet_v3_large", "convnext_tiny"):
        return [model.features[-1]]
    if backbone_name == "resnet50":
        return [model.layer4]
    if backbone_name == "densenet121":
        # Unfreeze the final dense block and its subsequent batch-norm together
        return [model.features.denseblock4, model.features.norm5]
    raise ValueError(f"Unknown backbone: {backbone_name!r}")


def build_model(backbone_name: str, num_classes: int, use_imagenet: bool = True):
    """Build a transfer-learning model with a backbone-specific classifier head."""

    if backbone_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if use_imagenet else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, num_classes),
        )

    elif backbone_name == "mobilenet_v3_large":
        weights = models.MobileNet_V3_Large_Weights.DEFAULT if use_imagenet else None
        model = models.mobilenet_v3_large(weights=weights)
        # Original head: Linear(960,1280) → Hardswish → Dropout(0.2) → Linear(1280,1000)
        # Keep the same structure but raise dropout and replace the final linear.
        in_features = model.classifier[0].in_features   # 960
        model.classifier = nn.Sequential(
            nn.Linear(in_features, 1280),
            nn.Hardswish(),
            nn.Dropout(p=0.4),
            nn.Linear(1280, num_classes),
        )

    elif backbone_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if use_imagenet else None
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features              # 2048
        model.fc = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, num_classes),
        )

    elif backbone_name == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT if use_imagenet else None
        model = models.densenet121(weights=weights)
        in_features = model.classifier.in_features      # 1024
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, num_classes),
        )

    elif backbone_name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if use_imagenet else None
        model = models.convnext_tiny(weights=weights)
        # Original head: LayerNorm2d(768) → Flatten → Linear(768,1000)
        # Keep the normalization and flatten; insert dropout before the new linear.
        layernorm = model.classifier[0]
        flatten   = model.classifier[1]
        in_features = model.classifier[2].in_features   # 768
        model.classifier = nn.Sequential(
            layernorm,
            flatten,
            nn.Dropout(p=0.4),
            nn.Linear(in_features, num_classes),
        )

    else:
        raise ValueError(
            f"Unknown backbone {backbone_name!r}. "
            f"Choose from: efficientnet_b0, mobilenet_v3_large, resnet50, densenet121, convnext_tiny"
        )

    return model


def freeze_backbone(model, backbone_name: str):
    """Freeze all parameters, then unfreeze the classification head (Stage 1)."""
    for p in model.parameters():
        p.requires_grad = False
    for p in _get_head_module(model, backbone_name).parameters():
        p.requires_grad = True


def unfreeze_last_blocks(model, backbone_name: str):
    """Unfreeze the last feature block(s) and the classification head (Stage 2)."""
    for block in _get_last_feature_blocks(model, backbone_name):
        for p in block.parameters():
            p.requires_grad = True
    for p in _get_head_module(model, backbone_name).parameters():
        p.requires_grad = True


def build_stage2_param_groups(model, backbone_name: str, backbone_lr: float, head_lr: float):
    """Build optimizer param groups with separate LRs for backbone and head."""
    backbone_params = []
    for block in _get_last_feature_blocks(model, backbone_name):
        backbone_params.extend(p for p in block.parameters() if p.requires_grad)

    head_params = [p for p in _get_head_module(model, backbone_name).parameters() if p.requires_grad]

    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": backbone_lr})
    if head_params:
        groups.append({"params": head_params, "lr": head_lr})
    return groups


# ── K-fold data loaders ─────────────────────────────────────────────────────

def get_loaders_kfold(
    data_dir,
    batch_size=16,
    img_size=224,
    n_splits=5,
    seed=42,
    mean=IMAGENET_MEAN,
    std=IMAGENET_STD,
):
    """
    Build stratified K-fold data loaders.

    Data-leakage prevention strategy
    ---------------------------------
    * Labels and split indices are derived from a reference dataset that loads
      NO images and applies NO transforms (just reads directory structure).
    * For each fold, a *separate* ImageFolder is created for training (with
      augmentation) and a *separate* one for validation (resize + normalise only).
      Subset() is then applied so each loader only ever sees its own indices.
    * An assertion verifies that training and validation index sets are disjoint.
    """
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
    ])

    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    # ── Use a no-transform reference dataset solely to obtain labels and metadata.
    #    No image is loaded at this point; ImageFolder reads targets from the
    #    directory structure during __init__, so no transform is needed.
    ref_dataset = datasets.ImageFolder(root=data_dir)
    labels   = np.array(ref_dataset.targets)
    classes  = ref_dataset.classes
    n_samples = len(ref_dataset)
    del ref_dataset  # free the reference object; actual data loading happens below

    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits   = list(splitter.split(np.arange(n_samples), labels))

    fold_loaders = []
    for train_idx, val_idx in splits:
        # Explicit guard: training and validation indices must be completely disjoint.
        overlap = set(train_idx) & set(val_idx)
        assert len(overlap) == 0, (
            f"Data leakage detected: {len(overlap)} indices appear in both "
            f"the training set and the validation set."
        )

        # Each fold gets its own ImageFolder so the correct transform is applied
        # per split.  Subset restricts iteration to the assigned indices only.
        train_dataset = datasets.ImageFolder(root=data_dir, transform=train_transform)
        val_dataset   = datasets.ImageFolder(root=data_dir, transform=val_transform)

        train_subset  = Subset(train_dataset, train_idx)
        val_subset    = Subset(val_dataset,   val_idx)

        train_loader = DataLoader(
            train_subset, batch_size=batch_size, shuffle=True,
            num_workers=0, pin_memory=False,
        )
        val_loader   = DataLoader(
            val_subset, batch_size=batch_size, shuffle=False,
            num_workers=0, pin_memory=False,
        )

        fold_loaders.append((train_loader, val_loader))

    return fold_loaders, len(classes), classes


# ── Training / evaluation utilities ────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds   = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device, return_predictions=False):
    model.eval()
    running_loss = 0.0
    correct  = 0
    total    = 0
    all_labels = []
    all_preds  = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * imgs.size(0)
            preds   = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            if return_predictions:
                all_labels.extend(labels.cpu().numpy().tolist())
                all_preds.extend(preds.cpu().numpy().tolist())

    avg_loss = running_loss / total
    avg_acc  = correct / total

    if return_predictions:
        return avg_loss, avg_acc, all_labels, all_preds
    return avg_loss, avg_acc


def save_checkpoint(path, model, optimizer, fold_idx, epoch, backbone, num_classes, val_loss, val_acc, stage):
    torch.save(
        {
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "fold":        fold_idx,
            "epoch":       epoch,
            "backbone":    backbone,
            "num_classes": num_classes,
            "val_loss":    val_loss,
            "val_acc":     val_acc,
            "stage":       stage,
        },
        path,
    )


def plot_loss_curves(train_losses, val_losses, fold_idx, backbone_name):
    plt.figure(figsize=(10, 5))
    epochs = range(1, len(train_losses) + 1)
    plt.plot(epochs, train_losses, "b-o", label="Train Loss", linewidth=2, markersize=5)
    plt.plot(epochs, val_losses,   "r-o", label="Val Loss",   linewidth=2, markersize=5)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{backbone_name} | Fold {fold_idx} – Training / Validation Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# Main training loop: freeze-then-fine-tune over all configured backbones

import argparse


def run_backbone(backbone_name, args, device):
    """Run full K-fold training for one backbone and return per-fold results."""

    print("\n" + "=" * 72)
    print(f"  BACKBONE: {backbone_name}  |  ImageNet pretrain: {not args.no_imagenet}")
    print("=" * 72)

    use_imagenet = not args.no_imagenet

    fold_loaders, num_classes, classes = get_loaders_kfold(
        data_dir   = args.data_dir,
        batch_size = args.batch_size,
        img_size   = args.img_size,
        n_splits   = args.n_splits,
        seed       = SEED,
        mean       = IMAGENET_MEAN,
        std        = IMAGENET_STD,
    )

    criterion    = nn.CrossEntropyLoss()
    fold_results = []

    for fold_idx, (train_loader, val_loader) in enumerate(fold_loaders, start=1):
        print("\n" + "-" * 60)
        print(f"  Fold {fold_idx}/{args.n_splits}")
        print("-" * 60)

        model = build_model(
            backbone_name = backbone_name,
            num_classes   = num_classes,
            use_imagenet  = use_imagenet,
        ).to(device)

        train_losses, val_losses = [], []
        best_val_loss = float("inf")
        best_ckpt = f"{args.save_path}_{backbone_name}_fold{fold_idx}.pth"

        # ── Stage 1: train head only (backbone frozen) ──────────────────────
        freeze_backbone(model, backbone_name)
        optimizer = optim.Adam(
            [p for p in model.parameters() if p.requires_grad],
            lr=args.head_lr, weight_decay=args.weight_decay,
        )

        for epoch in range(1, args.stage1_epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

            train_losses.append(train_loss)
            val_losses.append(val_loss)

            print(
                f"[Stage1][Epoch {epoch}/{args.stage1_epochs}] "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
            )

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                save_checkpoint(
                    path=best_ckpt, model=model, optimizer=optimizer,
                    fold_idx=fold_idx, epoch=epoch, backbone=backbone_name,
                    num_classes=num_classes, val_loss=val_loss, val_acc=val_acc,
                    stage="head-only",
                )

        # ── Stage 2: unfreeze last block(s) + head ───────────────────────────
        unfreeze_last_blocks(model, backbone_name)
        stage2_groups = build_stage2_param_groups(model, backbone_name, args.backbone_lr, args.head_lr)
        optimizer = optim.Adam(stage2_groups, weight_decay=args.weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", patience=3, factor=0.5, verbose=True,
        )

        es_counter = 0
        for epoch in range(1, args.stage2_epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)
            scheduler.step(val_loss)

            train_losses.append(train_loss)
            val_losses.append(val_loss)

            lr_info = ", ".join(f"group{i}={pg['lr']:.2e}" for i, pg in enumerate(optimizer.param_groups))
            print(
                f"[Stage2][Epoch {epoch}/{args.stage2_epochs}] "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
                f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | {lr_info}"
            )

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                es_counter    = 0
                save_checkpoint(
                    path=best_ckpt, model=model, optimizer=optimizer,
                    fold_idx=fold_idx, epoch=args.stage1_epochs + epoch,
                    backbone=backbone_name, num_classes=num_classes,
                    val_loss=val_loss, val_acc=val_acc, stage="fine-tune",
                )
            else:
                es_counter += 1
                if es_counter >= args.early_stop_patience:
                    print(
                        f"Early stopping at epoch {epoch} "
                        f"(no improvement for {args.early_stop_patience} epochs)"
                    )
                    break

        plot_loss_curves(train_losses, val_losses, fold_idx, backbone_name)

        # ── Evaluate best checkpoint on the fold validation set ──────────────
        checkpoint = torch.load(best_ckpt, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])

        test_loss, test_acc, y_true, y_pred = evaluate(
            model, val_loader, criterion, device, return_predictions=True,
        )
        macro_f1 = f1_score(y_true, y_pred, average="macro")

        print(
            f"Fold {fold_idx} best-ckpt  "
            f"test_loss={test_loss:.4f}, test_acc={test_acc:.4f}, macro_f1={macro_f1:.4f}"
        )
        print("Confusion Matrix:")
        print(confusion_matrix(y_true, y_pred))
        print("Classification Report:")
        print(classification_report(y_true, y_pred, target_names=classes, digits=4))

        fold_results.append({"fold": fold_idx, "acc": test_acc, "macro_f1": macro_f1})

    # ── Per-backbone K-fold summary ──────────────────────────────────────────
    accs = [r["acc"]      for r in fold_results]
    f1s  = [r["macro_f1"] for r in fold_results]

    print("\n" + "=" * 72)
    print(f"K-Fold Summary  |  backbone={backbone_name}")
    print("=" * 72)
    for r in fold_results:
        print(f"  Fold {r['fold']}: acc={r['acc']:.4f}, macro_f1={r['macro_f1']:.4f}")
    print(f"  Mean acc    : {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"  Mean macroF1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print("=" * 72)

    return fold_results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir",   type=str, default=DATA_DIR)
    parser.add_argument(
        "--backbones", type=str, nargs="+",
        default=BACKBONES_TO_RUN,
        choices=["efficientnet_b0", "mobilenet_v3_large", "resnet50", "densenet121", "convnext_tiny"],
        help="One or more backbone names to train and compare.",
    )
    parser.add_argument("--img-size",   type=int,   default=224)
    parser.add_argument("--batch-size", type=int,   default=16)
    parser.add_argument("--n-splits",   type=int,   default=5)

    parser.add_argument("--stage1-epochs",       type=int,   default=5)
    parser.add_argument("--stage2-epochs",       type=int,   default=20)
    parser.add_argument("--head-lr",             type=float, default=1e-3)
    parser.add_argument("--backbone-lr",         type=float, default=1e-4)
    parser.add_argument("--weight-decay",        type=float, default=1e-4)
    parser.add_argument("--early-stop-patience", type=int,   default=7)

    parser.add_argument("--save-path", type=str, default="./models_weights_tl")
    parser.add_argument(
        "--device", type=str,
        default=(
            "cuda" if torch.cuda.is_available()
            else ("mps" if torch.backends.mps.is_available() else "cpu")
        ),
    )
    parser.add_argument("--no-imagenet", action="store_true",
                        help="Disable ImageNet pretrained initialisation.")
    args = parser.parse_args([])

    if not os.path.isdir(args.data_dir):
        raise SystemExit(f"Dataset directory not found: {args.data_dir}")

    device = torch.device(args.device)
    print(f"Device: {device}")

    # ── Run K-fold for every requested backbone ──────────────────────────────
    all_results = {}  # backbone_name -> list of fold dicts
    for backbone_name in args.backbones:
        set_seed(SEED)
        all_results[backbone_name] = run_backbone(backbone_name, args, device)

    # ── Cross-backbone comparison summary ───────────────────────────────────
    print("\n" + "=" * 72)
    print("Cross-Backbone Comparison Summary")
    print("=" * 72)
    print(f"  {'Backbone':<22}  {'Mean Acc':>9}  {'±':>4}  {'Mean macroF1':>13}  {'±':>4}")
    print("  " + "-" * 60)
    for backbone_name, fold_results in all_results.items():
        accs = [r["acc"]      for r in fold_results]
        f1s  = [r["macro_f1"] for r in fold_results]
        print(
            f"  {backbone_name:<22}  {np.mean(accs):>9.4f}  {np.std(accs):>4.4f}"
            f"  {np.mean(f1s):>13.4f}  {np.std(f1s):>4.4f}"
        )
    print("=" * 72)

    return all_results


all_results = main()


In [ ]:
# Backbone comparison: bar charts for mean CV accuracy and macro-F1

def plot_backbone_comparison(all_results):
    """Bar charts comparing K-fold accuracy and macro-F1 across all backbones."""
    if not all_results:
        print("No results to plot. Run main() first.")
        return

    backbone_names = list(all_results.keys())
    mean_accs  = [np.mean([r["acc"]      for r in all_results[b]]) for b in backbone_names]
    std_accs   = [np.std( [r["acc"]      for r in all_results[b]]) for b in backbone_names]
    mean_f1s   = [np.mean([r["macro_f1"] for r in all_results[b]]) for b in backbone_names]
    std_f1s    = [np.std( [r["macro_f1"] for r in all_results[b]]) for b in backbone_names]

    x      = np.arange(len(backbone_names))
    width  = 0.35
    colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, means, stds, metric in zip(
        axes,
        [mean_accs, mean_f1s],
        [std_accs,  std_f1s],
        ["Mean CV Accuracy", "Mean CV Macro-F1"],
    ):
        bars = ax.bar(x, means, width=0.55, yerr=stds, capsize=5,
                      color=colors[: len(backbone_names)], edgecolor="black", linewidth=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels(backbone_names, rotation=20, ha="right", fontsize=9)
        ax.set_ylabel(metric)
        ax.set_title(f"{metric} across Backbones\n(5-fold cross-validation, lavender water-stress)")
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", alpha=0.3)
        for bar, val in zip(bars, means):
            ax.text(
                bar.get_x() + bar.get_width() / 2.0, bar.get_height() + 0.015,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8,
            )

    plt.tight_layout()
    plt.show()


# Use the results produced by main() in the previous cell
if "all_results" in globals() and all_results:
    plot_backbone_comparison(all_results)
else:
    print("Run the training cell first to populate all_results.")
